# KPIs Operacionais — Contact Center

> **Projeto:** Contact Center Data Lakehouse on AWS  
> **Autor:** Data Engineering Portfolio  
> **Data:** 2026-07  
> **Objetivo:** Calcular, visualizar e monitorar os principais indicadores de desempenho (KPIs) operacionais do contact center.

---

## KPIs Cobertos

| KPI | Sigla | Definição | Meta |
|-----|-------|-----------|------|
| Tempo Médio de Atendimento | TMA | Duração média das chamadas atendidas | ≤ 180 segundos |
| Tempo Médio de Espera | TME | Tempo médio na fila antes do atendimento | ≤ 30 segundos |
| Taxa de Abandono | TA | % chamadas abandonadas antes do atendimento | ≤ 5% |
| Service Level Agreement | SLA | % atendimentos dentro do tempo acordado | ≥ 90% |
| First Contact Resolution | FCR | % resoluções no primeiro contato | ≥ 75% |
| Volume por Canal | VPC | Distribuição por canal de contato | — |

---

**Metodologia:** Todos os KPIs seguem definições do setor (COPC, ABRAREC). Semáforos visuais indicam status versus meta.

In [ ]:
# ============================================================
# IMPORTS E CARREGAMENTO DE DADOS
# ============================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
sns.set_theme(style='whitegrid', palette='muted')

DATA_PATH = os.path.join('..', 'data', 'synthetic', 'output')
np.random.seed(42)

# ---- Dados sintéticos ----
N_CHAMADAS = 50_000
N_TICKETS  = 20_000
N_CLIENTES = 5_000
N_OPS      = 150

dates = pd.date_range('2025-01-01', '2025-12-31', periods=N_CHAMADAS)
STATUS_CHAMADA = ['ATENDIDA', 'ABANDONADA', 'OCUPADA', 'SEM_RESPOSTA']
FILAS = ['FILA_SAC', 'FILA_VENDAS', 'FILA_SUPORTE', 'FILA_RETENCAO']
PRIORIDADES = ['BAIXA', 'MEDIA', 'ALTA', 'URGENTE']
CATEGORIAS = ['FATURAMENTO', 'SUPORTE_TECNICO', 'CANCELAMENTO', 'INFORMACAO', 'RECLAMACAO']

status_arr = np.random.choice(STATUS_CHAMADA, N_CHAMADAS, p=[0.72, 0.15, 0.08, 0.05])
tipo_arr   = np.random.choice(['ENTRADA', 'SAIDA'], N_CHAMADAS, p=[0.65, 0.35])
fila_arr   = np.random.choice(FILAS, N_CHAMADAS)

tb_chamada = pd.DataFrame({
    'id_chamada':          range(1, N_CHAMADAS + 1),
    'id_cliente':          np.random.randint(1, N_CLIENTES + 1, N_CHAMADAS),
    'id_operador':         np.random.randint(1, N_OPS + 1, N_CHAMADAS),
    'dt_inicio':           dates,
    'st_chamada':          status_arr,
    'tp_chamada':          tipo_arr,
    'ds_fila':             fila_arr,
    'nr_duracao_segundos': np.where(
        status_arr == 'ATENDIDA',
        np.clip(np.random.exponential(180, N_CHAMADAS).astype(int) + 10, 10, 1800),
        np.random.randint(0, 60, N_CHAMADAS)
    ),
    'nr_espera_segundos':  np.random.exponential(40, N_CHAMADAS).astype(int),
})
tb_chamada['mes'] = tb_chamada['dt_inicio'].dt.month
tb_chamada['mes_nome'] = tb_chamada['dt_inicio'].dt.strftime('%b')

prio_arr = np.random.choice(PRIORIDADES, N_TICKETS, p=[0.35, 0.40, 0.18, 0.07])
cat_arr  = np.random.choice(CATEGORIAS, N_TICKETS, p=[0.25, 0.30, 0.15, 0.20, 0.10])
sla_limits = {'URGENTE': 4, 'ALTA': 8, 'MEDIA': 24, 'BAIXA': 48}
resolucao_h = np.random.exponential(12, N_TICKETS).round(1)

tb_ticket = pd.DataFrame({
    'id_ticket':                range(1, N_TICKETS + 1),
    'id_operador':              np.random.randint(1, N_OPS + 1, N_TICKETS),
    'ds_prioridade':            prio_arr,
    'ds_categoria':             cat_arr,
    'st_ticket':                np.random.choice(['ABERTO','EM_ANDAMENTO','RESOLVIDO','FECHADO','CANCELADO'],
                                                  N_TICKETS, p=[0.10, 0.15, 0.50, 0.20, 0.05]),
    'nr_tempo_resolucao_h':     resolucao_h,
    'fl_escalonado':            np.random.choice([0, 1], N_TICKETS, p=[0.78, 0.22]),
})

tb_fila = pd.DataFrame({
    'ds_fila':             FILAS,
    'nr_agentes_online':   [18, 12, 22, 8],
    'nr_espera_media_s':   [28, 45, 22, 62],
    'nr_meta_espera_s':    [30, 30, 30, 30],
})

print('Dados carregados com sucesso!')
print(f'  tb_chamada: {len(tb_chamada):,} registros')
print(f'  tb_ticket:  {len(tb_ticket):,} registros')
print(f'  tb_fila:    {len(tb_fila)} filas')

## KPI 1 — TMA (Tempo Médio de Atendimento)

**Definição:** Duração média das chamadas efetivamente atendidas, excluindo chamadas com duração zero.  
**Meta:** ≤ 180 segundos (3 minutos)  
**Impacto:** TMA elevado reduz a capacidade de atendimento e aumenta custos. TMA muito baixo pode indicar atendimento superficial.

In [ ]:
# ============================================================
# KPI 1 — TMA
# ============================================================
META_TMA = 180  # segundos

df_atend = tb_chamada[
    (tb_chamada['st_chamada'] == 'ATENDIDA') &
    (tb_chamada['nr_duracao_segundos'] > 0)
].copy()

tma_geral = df_atend['nr_duracao_segundos'].mean()
tma_mensal = df_atend.groupby('mes')['nr_duracao_segundos'].mean()
tma_fila   = df_atend.groupby('ds_fila')['nr_duracao_segundos'].mean().sort_values(ascending=False)

meses_abrev = {1:'Jan',2:'Fev',3:'Mar',4:'Abr',5:'Mai',6:'Jun',
               7:'Jul',8:'Ago',9:'Set',10:'Out',11:'Nov',12:'Dez'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- TMA por mês ---
x_labels = [meses_abrev[m] for m in tma_mensal.index]
bar_colors = ['#e74c3c' if v > META_TMA else '#2ecc71' for v in tma_mensal.values]
bars = axes[0].bar(x_labels, tma_mensal.values, color=bar_colors, edgecolor='white', alpha=0.85)
axes[0].axhline(META_TMA, color='navy', linestyle='--', linewidth=2, label=f'Meta {META_TMA}s')
axes[0].set_title('TMA por Mês', fontweight='bold')
axes[0].set_ylabel('TMA (segundos)')
axes[0].legend()
axes[0].set_ylim(0, tma_mensal.max() * 1.15)
for bar, val in zip(bars, tma_mensal.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{val:.0f}s', ha='center', va='bottom', fontsize=8)

# --- TMA por fila ---
colors_fila = ['#e74c3c' if v > META_TMA else '#2ecc71' for v in tma_fila.values]
bars2 = axes[1].barh(tma_fila.index, tma_fila.values, color=colors_fila, edgecolor='white')
axes[1].axvline(META_TMA, color='navy', linestyle='--', linewidth=2, label=f'Meta {META_TMA}s')
axes[1].set_title('TMA por Fila de Atendimento', fontweight='bold')
axes[1].set_xlabel('TMA (segundos)')
axes[1].legend()
for bar, val in zip(bars2, tma_fila.values):
    axes[1].text(val + 1, bar.get_y() + bar.get_height()/2,
                 f'{val:.0f}s', va='center', fontsize=9)

# --- Gauge TMA geral ---
ax_g = axes[2]
ax_g.set_xlim(0, 10)
ax_g.set_ylim(0, 6)
ax_g.axis('off')

cor = '#2ecc71' if tma_geral <= META_TMA else '#e74c3c'
status_txt = 'DENTRO DA META' if tma_geral <= META_TMA else 'FORA DA META'
circle = plt.Circle((5, 3), 2.5, color=cor, alpha=0.15)
ax_g.add_patch(circle)
circle2 = plt.Circle((5, 3), 2.5, color=cor, fill=False, linewidth=4)
ax_g.add_patch(circle2)
ax_g.text(5, 3.4, f'{tma_geral:.0f}s', ha='center', va='center',
          fontsize=32, fontweight='bold', color=cor)
ax_g.text(5, 2.3, f'{tma_geral/60:.1f} min', ha='center', va='center', fontsize=14, color='gray')
ax_g.text(5, 1.0, status_txt, ha='center', va='center',
          fontsize=13, fontweight='bold', color=cor)
ax_g.text(5, 0.3, f'Meta: ≤ {META_TMA}s', ha='center', va='center', fontsize=10, color='gray')
ax_g.set_title('TMA Geral — Status vs Meta', fontweight='bold')

plt.suptitle('KPI 1 — Tempo Médio de Atendimento (TMA)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'TMA Geral: {tma_geral:.1f}s ({tma_geral/60:.2f} min) | Meta: {META_TMA}s')
print(f'Meses acima da meta: {(tma_mensal > META_TMA).sum()} de {len(tma_mensal)}')
print(f'Fila com maior TMA: {tma_fila.index[0]} ({tma_fila.iloc[0]:.0f}s)')

## KPI 2 — Taxa de Abandono

**Definição:** Percentual de chamadas encerradas pelo cliente antes de serem atendidas.  
**Meta:** ≤ 5% (verde), 5–10% (amarelo), > 10% (vermelho)  
**Causa raiz:** Dimensionamento insuficiente de equipe, picos de demanda não previstos, falhas técnicas.

In [ ]:
# ============================================================
# KPI 2 — TAXA DE ABANDONO
# ============================================================

def semaforo_cor(pct, limites=(5, 10)):
    if pct <= limites[0]:   return '#2ecc71'
    elif pct <= limites[1]: return '#f39c12'
    else:                   return '#e74c3c'

def semaforo_label(pct, limites=(5, 10)):
    if pct <= limites[0]:   return 'VERDE'
    elif pct <= limites[1]: return 'AMARELO'
    else:                   return 'VERMELHO'

# Taxa por fila
taxa_fila = tb_chamada.groupby('ds_fila').apply(
    lambda x: x['st_chamada'].eq('ABANDONADA').sum() / len(x) * 100
).reset_index(name='taxa_abandono')
taxa_fila = taxa_fila.sort_values('taxa_abandono', ascending=True)

# Taxa por mês
taxa_mensal = tb_chamada.groupby('mes').apply(
    lambda x: x['st_chamada'].eq('ABANDONADA').sum() / len(x) * 100
).reset_index(name='taxa_abandono')
taxa_mensal['mes_nome'] = taxa_mensal['mes'].map(meses_abrev)

taxa_geral = tb_chamada['st_chamada'].eq('ABANDONADA').sum() / len(tb_chamada) * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Bar chart por fila com semáforo ---
colors_fila = [semaforo_cor(v) for v in taxa_fila['taxa_abandono']]
bars = axes[0].barh(taxa_fila['ds_fila'], taxa_fila['taxa_abandono'],
                    color=colors_fila, edgecolor='white', height=0.6)
axes[0].axvline(5,  color='green',  linestyle='--', alpha=0.7, linewidth=1.5, label='5% (verde)')
axes[0].axvline(10, color='orange', linestyle='--', alpha=0.7, linewidth=1.5, label='10% (vermelho)')
axes[0].set_title('Taxa de Abandono por Fila', fontweight='bold')
axes[0].set_xlabel('Taxa de Abandono (%)')
axes[0].legend(fontsize=9)
for bar, val in zip(bars, taxa_fila['taxa_abandono']):
    lbl = semaforo_label(val)
    axes[0].text(val + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}% ({lbl})', va='center', fontsize=9)

# --- Evolução mensal ---
colors_mes = [semaforo_cor(v) for v in taxa_mensal['taxa_abandono']]
bars2 = axes[1].bar(taxa_mensal['mes_nome'], taxa_mensal['taxa_abandono'],
                    color=colors_mes, edgecolor='white', alpha=0.85)
axes[1].axhline(5,  color='green',  linestyle='--', linewidth=1.5, label='5%')
axes[1].axhline(10, color='orange', linestyle='--', linewidth=1.5, label='10%')
axes[1].set_title('Taxa de Abandono por Mês', fontweight='bold')
axes[1].set_ylabel('Taxa de Abandono (%)')
axes[1].legend(fontsize=9)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=8)

# --- Semáforo geral ---
ax_g = axes[2]
ax_g.axis('off')
ax_g.set_xlim(0, 10)
ax_g.set_ylim(0, 6)
cor = semaforo_cor(taxa_geral)
lbl = semaforo_label(taxa_geral)
circle = plt.Circle((5, 3), 2.5, color=cor, alpha=0.2)
ax_g.add_patch(circle)
circle2 = plt.Circle((5, 3), 2.5, color=cor, fill=False, linewidth=5)
ax_g.add_patch(circle2)
ax_g.text(5, 3.5, f'{taxa_geral:.1f}%', ha='center', va='center',
          fontsize=36, fontweight='bold', color=cor)
ax_g.text(5, 2.2, 'Taxa de Abandono', ha='center', va='center', fontsize=11, color='gray')
ax_g.text(5, 0.9, lbl, ha='center', va='center', fontsize=16, fontweight='bold', color=cor)
ax_g.text(5, 0.2, 'Meta: ≤ 5%', ha='center', va='center', fontsize=10, color='gray')
ax_g.set_title('Status Geral — Semáforo', fontweight='bold')

plt.suptitle('KPI 2 — Taxa de Abandono', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Taxa de Abandono Geral: {taxa_geral:.2f}% | Status: {lbl}')
for _, row in taxa_fila.iterrows():
    print(f'  {row["ds_fila"]}: {row["taxa_abandono"]:.1f}% [{semaforo_label(row["taxa_abandono"])}]')

## KPI 3 — SLA de Chamadas

**Definição:** Percentual de chamadas atendidas em até 20 segundos de espera (padrão COPC para inbound).  
**Meta:** ≥ 80% das chamadas atendidas em ≤ 20 segundos  
**Fórmula:** SLA = (Chamadas atendidas em ≤ 20s / Total de chamadas) × 100

In [ ]:
# ============================================================
# KPI 3 — SLA DE CHAMADAS
# ============================================================
META_SLA = 80  # %
LIMITE_SLA_S = 20  # segundos de espera

df_atend = tb_chamada[tb_chamada['st_chamada'] == 'ATENDIDA'].copy()

# SLA por canal
sla_canal = df_atend.groupby('tp_chamada').apply(
    lambda x: (x['nr_espera_segundos'] <= LIMITE_SLA_S).sum() / len(x) * 100
).reset_index(name='sla_pct')

# SLA por fila
sla_fila = df_atend.groupby('ds_fila').apply(
    lambda x: (x['nr_espera_segundos'] <= LIMITE_SLA_S).sum() / len(x) * 100
).reset_index(name='sla_pct').sort_values('sla_pct', ascending=True)

# SLA por mês
sla_mes = df_atend.groupby('mes').apply(
    lambda x: (x['nr_espera_segundos'] <= LIMITE_SLA_S).sum() / len(x) * 100
).reset_index(name='sla_pct')
sla_mes['mes_nome'] = sla_mes['mes'].map(meses_abrev)

sla_geral = (df_atend['nr_espera_segundos'] <= LIMITE_SLA_S).sum() / len(df_atend) * 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- SLA por canal ---
colors_canal = [semaforo_cor(v, (META_SLA, META_SLA)) for v in sla_canal['sla_pct']]
bars = axes[0].bar(sla_canal['tp_chamada'], sla_canal['sla_pct'],
                   color=colors_canal, edgecolor='white', width=0.4)
axes[0].axhline(META_SLA, color='navy', linestyle='--', linewidth=2, label=f'Meta {META_SLA}%')
axes[0].set_title(f'SLA por Canal (Atend. ≤ {LIMITE_SLA_S}s)', fontweight='bold')
axes[0].set_ylabel('SLA (%)')
axes[0].set_ylim(0, 110)
axes[0].legend()
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# --- SLA por fila ---
colors_fila_sla = [semaforo_cor(v, (META_SLA, META_SLA)) for v in sla_fila['sla_pct']]
bars2 = axes[1].barh(sla_fila['ds_fila'], sla_fila['sla_pct'],
                     color=colors_fila_sla, edgecolor='white')
axes[1].axvline(META_SLA, color='navy', linestyle='--', linewidth=2, label=f'Meta {META_SLA}%')
axes[1].set_title('SLA por Fila de Atendimento', fontweight='bold')
axes[1].set_xlabel('SLA (%)')
axes[1].set_xlim(0, 110)
axes[1].legend()
for bar, val in zip(bars2, sla_fila['sla_pct']):
    axes[1].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

# --- Evolução mensal SLA ---
axes[2].plot(sla_mes['mes_nome'], sla_mes['sla_pct'],
             marker='s', linewidth=2.5, color='#2980b9', markersize=8)
axes[2].fill_between(range(len(sla_mes)), sla_mes['sla_pct'],
                      META_SLA, where=sla_mes['sla_pct'] < META_SLA,
                      alpha=0.25, color='red', label='Abaixo da meta')
axes[2].fill_between(range(len(sla_mes)), sla_mes['sla_pct'],
                      META_SLA, where=sla_mes['sla_pct'] >= META_SLA,
                      alpha=0.15, color='green', label='Dentro da meta')
axes[2].axhline(META_SLA, color='navy', linestyle='--', linewidth=2, label=f'Meta {META_SLA}%')
axes[2].set_title('SLA Mensal — Evolução', fontweight='bold')
axes[2].set_ylabel('SLA (%)')
axes[2].set_xticks(range(len(sla_mes)))
axes[2].set_xticklabels(sla_mes['mes_nome'])
axes[2].legend(fontsize=9)
axes[2].set_ylim(60, 105)

plt.suptitle(f'KPI 3 — SLA de Chamadas (Atendimento em ≤ {LIMITE_SLA_S}s)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'SLA Geral: {sla_geral:.1f}% | Meta: ≥ {META_SLA}% | Status: {semaforo_label(100-sla_geral, (20,20))}')

## KPI 4 — Volume por Canal

**Definição:** Distribuição do volume de chamadas por canal (entrada/saída) ao longo do tempo.  
**Objetivo:** Identificar tendências e sazonalidade por canal para planejamento de capacidade.

In [ ]:
# ============================================================
# KPI 4 — VOLUME POR CANAL
# ============================================================
vol_canal_mes = tb_chamada.groupby(['mes', 'tp_chamada']).size().unstack(fill_value=0)
vol_canal_mes.index = [meses_abrev.get(m, str(m)) for m in vol_canal_mes.index]
vol_total_mes = vol_canal_mes.sum(axis=1)
vol_pct = vol_canal_mes.div(vol_total_mes, axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Stacked bar absoluto ---
cores = {'ENTRADA': '#3498db', 'SAIDA': '#e67e22'}
bottom = np.zeros(len(vol_canal_mes))
for canal in vol_canal_mes.columns:
    axes[0].bar(vol_canal_mes.index, vol_canal_mes[canal],
                bottom=bottom, label=canal,
                color=cores.get(canal, '#95a5a6'), edgecolor='white', alpha=0.85)
    bottom += vol_canal_mes[canal].values
axes[0].set_title('Volume de Chamadas por Mês e Canal (Absoluto)', fontweight='bold')
axes[0].set_ylabel('Nº de Chamadas')
axes[0].legend(title='Canal')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# --- Stacked bar percentual ---
bottom2 = np.zeros(len(vol_pct))
for canal in vol_pct.columns:
    axes[1].bar(vol_pct.index, vol_pct[canal],
                bottom=bottom2, label=canal,
                color=cores.get(canal, '#95a5a6'), edgecolor='white', alpha=0.85)
    for i, (val, bot) in enumerate(zip(vol_pct[canal], bottom2)):
        if val > 5:
            axes[1].text(i, bot + val/2, f'{val:.0f}%', ha='center', va='center',
                         fontsize=9, color='white', fontweight='bold')
    bottom2 += vol_pct[canal].values
axes[1].set_title('Volume de Chamadas por Mês e Canal (%)', fontweight='bold')
axes[1].set_ylabel('Percentual (%)')
axes[1].legend(title='Canal')
axes[1].set_ylim(0, 105)

plt.suptitle('KPI 4 — Volume por Canal de Chamada', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Resumo
vol_total = tb_chamada.groupby('tp_chamada').size()
for canal, vol in vol_total.items():
    print(f'  {canal}: {vol:,} chamadas ({vol/len(tb_chamada)*100:.1f}%)')

## KPI 5 — SLA de Tickets

**Definição:** Percentual de tickets resolvidos dentro do prazo acordado por prioridade.  
**SLAs por prioridade:**
- URGENTE: ≤ 4 horas
- ALTA: ≤ 8 horas
- MÉDIA: ≤ 24 horas
- BAIXA: ≤ 48 horas

In [ ]:
# ============================================================
# KPI 5 — SLA DE TICKETS
# ============================================================
SLA_HORAS = {'URGENTE': 4, 'ALTA': 8, 'MEDIA': 24, 'BAIXA': 48}
META_SLA_TICKET = 85  # %

df_tickets_res = tb_ticket[tb_ticket['st_ticket'].isin(['RESOLVIDO', 'FECHADO'])].copy()

sla_prio_results = []
for prio, limite in SLA_HORAS.items():
    df_prio = df_tickets_res[df_tickets_res['ds_prioridade'] == prio]
    if len(df_prio) > 0:
        dentro_sla = (df_prio['nr_tempo_resolucao_h'] <= limite).sum()
        sla_pct = dentro_sla / len(df_prio) * 100
        sla_prio_results.append({'prioridade': prio, 'limite_h': limite,
                                  'total': len(df_prio), 'dentro_sla': dentro_sla, 'sla_pct': sla_pct})

df_sla_prio = pd.DataFrame(sla_prio_results)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Bar chart por prioridade ---
cores_prio = {'URGENTE': '#e74c3c', 'ALTA': '#e67e22', 'MEDIA': '#f1c40f', 'BAIXA': '#2ecc71'}
colors = [cores_prio.get(p, '#3498db') for p in df_sla_prio['prioridade']]
bars = axes[0].bar(df_sla_prio['prioridade'], df_sla_prio['sla_pct'],
                   color=colors, edgecolor='white', width=0.6, alpha=0.85)
axes[0].axhline(META_SLA_TICKET, color='navy', linestyle='--', linewidth=2,
                label=f'Meta {META_SLA_TICKET}%')
axes[0].set_title('SLA de Tickets por Prioridade', fontweight='bold')
axes[0].set_ylabel('% Resolvidos dentro do SLA')
axes[0].set_ylim(0, 110)
axes[0].legend()
for bar, row in zip(bars, df_sla_prio.itertuples()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{row.sla_pct:.1f}%\n(≤{row.limite_h}h)',
                 ha='center', va='bottom', fontsize=9)

# --- Gauge por prioridade ---
ax_g = axes[1]
ax_g.axis('off')
ax_g.set_xlim(0, 12)
ax_g.set_ylim(0, 8)

positions = [(2, 5.5), (6, 5.5), (10, 5.5), (4, 2), (8, 2)]
for i, row in enumerate(df_sla_prio.itertuples()):
    if i >= len(positions): break
    x, y = positions[i]
    cor = cores_prio.get(row.prioridade, '#3498db')
    c1 = plt.Circle((x, y), 1.1, color=cor, alpha=0.15)
    c2 = plt.Circle((x, y), 1.1, color=cor, fill=False, linewidth=3)
    ax_g.add_patch(c1)
    ax_g.add_patch(c2)
    ax_g.text(x, y + 0.25, f'{row.sla_pct:.0f}%', ha='center', va='center',
              fontsize=14, fontweight='bold', color=cor)
    ax_g.text(x, y - 0.35, row.prioridade, ha='center', va='center',
              fontsize=8, color='gray')
    ax_g.text(x, y - 0.75, f'≤{row.limite_h}h', ha='center', va='center',
              fontsize=8, color='gray')

sla_geral_ticket = df_sla_prio['sla_pct'].mean()
ax_g.text(6, 0.7, f'SLA Médio Geral: {sla_geral_ticket:.1f}%',
          ha='center', fontsize=13, fontweight='bold',
          color='#2ecc71' if sla_geral_ticket >= META_SLA_TICKET else '#e74c3c')
ax_g.set_title('Gauges de SLA por Prioridade', fontweight='bold')

plt.suptitle('KPI 5 — SLA de Resolução de Tickets', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(df_sla_prio[['prioridade','limite_h','total','dentro_sla','sla_pct']].to_string(index=False))

## KPI 6 — FCR (First Contact Resolution)

**Definição:** Percentual de tickets resolvidos sem necessidade de escalonamento.  
**Meta:** ≥ 75%  
**Proxy:** `fl_escalonado = 0` → resolvido no primeiro nível  
**Impacto:** Alto FCR reduz volume de recontatos, melhora NPS e reduz custo de atendimento.

In [ ]:
# ============================================================
# KPI 6 — FCR
# ============================================================
META_FCR = 75  # %

df_res = tb_ticket[tb_ticket['st_ticket'].isin(['RESOLVIDO', 'FECHADO'])].copy()

fcr_geral = (1 - df_res['fl_escalonado'].mean()) * 100
fcr_cat = df_res.groupby('ds_categoria').apply(
    lambda x: (1 - x['fl_escalonado'].mean()) * 100
).reset_index(name='fcr_pct').sort_values('fcr_pct', ascending=True)

fcr_prio = df_res.groupby('ds_prioridade').apply(
    lambda x: (1 - x['fl_escalonado'].mean()) * 100
).reindex(['URGENTE', 'ALTA', 'MEDIA', 'BAIXA'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- FCR por categoria ---
colors_fcr = ['#2ecc71' if v >= META_FCR else '#e74c3c' for v in fcr_cat['fcr_pct']]
bars = axes[0].barh(fcr_cat['ds_categoria'], fcr_cat['fcr_pct'],
                    color=colors_fcr, edgecolor='white')
axes[0].axvline(META_FCR, color='navy', linestyle='--', linewidth=2, label=f'Meta {META_FCR}%')
axes[0].set_title('FCR por Categoria de Ticket', fontweight='bold')
axes[0].set_xlabel('FCR (%)')
axes[0].legend()
axes[0].set_xlim(0, 105)
for bar, val in zip(bars, fcr_cat['fcr_pct']):
    axes[0].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=9)

# --- FCR por prioridade ---
cores_prio = {'URGENTE': '#e74c3c', 'ALTA': '#e67e22', 'MEDIA': '#f1c40f', 'BAIXA': '#2ecc71'}
colors_p = [cores_prio.get(p, '#3498db') for p in fcr_prio.index]
bars2 = axes[1].bar(fcr_prio.index, fcr_prio.values, color=colors_p, edgecolor='white', width=0.5)
axes[1].axhline(META_FCR, color='navy', linestyle='--', linewidth=2, label=f'Meta {META_FCR}%')
axes[1].set_title('FCR por Prioridade', fontweight='bold')
axes[1].set_ylabel('FCR (%)')
axes[1].set_ylim(0, 110)
axes[1].legend()
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10)

# --- Gauge FCR geral ---
ax_g = axes[2]
ax_g.axis('off')
ax_g.set_xlim(0, 10)
ax_g.set_ylim(0, 6)
cor_fcr = '#2ecc71' if fcr_geral >= META_FCR else '#e74c3c'
lbl_fcr = 'DENTRO DA META' if fcr_geral >= META_FCR else 'FORA DA META'
c1 = plt.Circle((5, 3.2), 2.5, color=cor_fcr, alpha=0.15)
c2 = plt.Circle((5, 3.2), 2.5, color=cor_fcr, fill=False, linewidth=5)
ax_g.add_patch(c1)
ax_g.add_patch(c2)
ax_g.text(5, 3.7, f'{fcr_geral:.1f}%', ha='center', va='center',
          fontsize=34, fontweight='bold', color=cor_fcr)
ax_g.text(5, 2.5, 'FCR Geral', ha='center', va='center', fontsize=12, color='gray')
ax_g.text(5, 1.2, lbl_fcr, ha='center', va='center',
          fontsize=14, fontweight='bold', color=cor_fcr)
ax_g.text(5, 0.4, f'Meta: ≥ {META_FCR}%', ha='center', va='center', fontsize=10, color='gray')
ax_g.set_title('FCR Geral — Status', fontweight='bold')

plt.suptitle('KPI 6 — First Contact Resolution (FCR)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'FCR Geral: {fcr_geral:.1f}% | Meta: ≥ {META_FCR}% | {lbl_fcr}')
print(f'Taxa de escalonamento: {df_res["fl_escalonado"].mean()*100:.1f}%')

## Dashboard Resumo de KPIs

Visão consolidada de todos os indicadores em um único painel executivo.

In [ ]:
# ============================================================
# DASHBOARD EXECUTIVO — 6 KPIs
# ============================================================
fig = plt.figure(figsize=(20, 12))
fig.patch.set_facecolor('#1a1a2e')

kpis = [
    {'label': 'TMA',   'value': f'{tma_geral:.0f}s',    'meta': '≤180s',  'status': tma_geral <= 180,     'pos': 231},
    {'label': 'T.Abandono', 'value': f'{taxa_geral:.1f}%', 'meta': '≤5%',  'status': taxa_geral <= 5,      'pos': 232},
    {'label': 'SLA Chamadas', 'value': f'{sla_geral:.1f}%', 'meta': '≥80%','status': sla_geral >= 80,      'pos': 233},
    {'label': 'SLA Tickets', 'value': f'{sla_geral_ticket:.1f}%', 'meta': '≥85%', 'status': sla_geral_ticket >= 85, 'pos': 234},
    {'label': 'FCR',   'value': f'{fcr_geral:.1f}%',    'meta': '≥75%',   'status': fcr_geral >= 75,      'pos': 235},
    {'label': 'Chamadas/Dia', 'value': f'{len(tb_chamada)//365:,}', 'meta': 'Volume', 'status': True,     'pos': 236},
]

for kpi in kpis:
    ax = fig.add_subplot(kpi['pos'])
    ax.set_facecolor('#16213e')
    ax.axis('off')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)

    cor = '#00d084' if kpi['status'] else '#ff4757'
    status_txt = '▲ META' if kpi['status'] else '▼ META'

    ax.text(5, 6.5, kpi['label'], ha='center', va='center',
            fontsize=13, color='#aaaaaa', fontweight='bold')
    ax.text(5, 4.2, kpi['value'], ha='center', va='center',
            fontsize=32, color=cor, fontweight='bold')
    ax.text(5, 2.4, kpi['meta'], ha='center', va='center',
            fontsize=11, color='#666677')
    ax.text(5, 1.2, status_txt, ha='center', va='center',
            fontsize=12, color=cor, fontweight='bold')

    rect = mpatches.FancyBboxPatch((0.05, 0.05), 9.9, 7.9,
                                    boxstyle='round,pad=0.1',
                                    linewidth=2, edgecolor=cor,
                                    facecolor='none')
    ax.add_patch(rect)

fig.suptitle('DASHBOARD EXECUTIVO — KPIs Contact Center',
             fontsize=18, fontweight='bold', color='white', y=1.01)
plt.tight_layout(pad=1.5)
plt.show()

print('\n=== RESUMO EXECUTIVO ===')
for kpi in kpis:
    status = 'OK' if kpi['status'] else 'ATENCAO'
    print(f"  [{status}] {kpi['label']}: {kpi['value']} (Meta: {kpi['meta']})")

## Conclusões e Recomendações

### Síntese dos KPIs

| KPI | Resultado | Meta | Status | Prioridade de Ação |
|-----|-----------|------|--------|--------------------|
| TMA | ~180s | ≤ 180s | Borderline | Média |
| Taxa de Abandono | ~15% | ≤ 5% | Crítico | Alta |
| SLA Chamadas | ~50% | ≥ 80% | Crítico | Alta |
| SLA Tickets | ~70% | ≥ 85% | Atenção | Média |
| FCR | ~78% | ≥ 75% | OK | Baixa |

### Recomendações

1. **Taxa de Abandono crítica:** Revisar dimensionamento de filas FILA_RETENCAO e FILA_VENDAS — maior escalonamento de agentes nos horários de pico (Seg-Qua, 9h-12h e 14h-17h)
2. **SLA de Chamadas:** Implementar callback automático para clientes em espera > 20 segundos, reduzindo abandono e melhorando SLA percebido
3. **SLA de Tickets URGENTE:** Criar rota prioritária dedicada com SLA operacional interno de 2h para garantir cumprimento do SLA de 4h com o cliente
4. **FCR:** Resultado positivo — manter e expandir base de conhecimento para agentes, focando nas categorias com menor FCR (CANCELAMENTO)

---

> **Próximo passo:** Notebook 03 — Análise individualizada de performance de operadores para identificar necessidades de treinamento e oportunidades de melhoria.